In [97]:
%pip install pandas

import pandas as pd



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [98]:
# World Bank GDP data
df = pd.read_csv('./API_NY.GDP.MKTP.CD_DS2_en_csv_v2_133326.csv', skiprows=4)

# Drop the trailing unnamed column World Bank appends
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

df


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Aruba,ABW,GDP (current US$),NY.GDP.MKTP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,2.983637e+09,3.092428e+09,3.276188e+09,3.346623e+09,2.471419e+09,2.880903e+09,3.324034e+09,3.834730e+09,4.265651e+09,NaN
1,Africa Eastern and Southern,AFE,GDP (current US$),NY.GDP.MKTP.CD,2.420569e+10,2.495889e+10,2.707323e+10,3.176914e+10,3.027955e+10,3.380618e+10,...,8.318681e+11,9.780765e+11,1.020956e+12,1.018715e+12,9.386076e+11,1.114145e+12,1.228968e+12,1.179359e+12,1.242694e+12,NaN
2,Afghanistan,AFG,GDP (current US$),NY.GDP.MKTP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,1.811657e+10,1.875346e+10,1.805322e+10,1.879944e+10,1.995593e+10,1.426000e+10,1.449724e+10,1.715223e+10,NaN,NaN
3,Africa Western and Central,AFW,GDP (current US$),NY.GDP.MKTP.CD,1.190481e+10,1.270772e+10,1.363059e+10,1.446891e+10,1.580356e+10,1.692088e+10,...,7.000282e+11,6.940513e+11,7.778403e+11,1.026996e+12,9.637847e+11,1.026651e+12,1.063649e+12,9.382384e+11,7.363850e+11,NaN
4,Angola,AGO,GDP (current US$),NY.GDP.MKTP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,5.987825e+10,8.437694e+10,8.951279e+10,8.073443e+10,5.885246e+10,7.955954e+10,1.312122e+11,1.071677e+11,1.009989e+11,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,Kosovo,XKX,GDP (current US$),NY.GDP.MKTP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,6.682677e+09,7.180769e+09,7.878763e+09,7.899741e+09,7.717143e+09,9.413409e+09,9.354908e+09,1.046822e+10,1.119725e+10,NaN
262,"Yemen, Rep.",YEM,GDP (current US$),NY.GDP.MKTP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,3.131783e+10,2.684223e+10,2.160616e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
263,South Africa,ZAF,GDP (current US$),NY.GDP.MKTP.CD,8.748597e+09,9.225996e+09,9.813996e+09,1.085420e+10,1.195600e+10,1.306899e+10,...,3.235855e+11,3.814488e+11,4.052607e+11,3.893300e+11,3.379747e+11,4.199863e+11,4.075960e+11,3.814407e+11,4.011450e+11,NaN
264,Zambia,ZMB,GDP (current US$),NY.GDP.MKTP.CD,6.987397e+08,6.823597e+08,6.792797e+08,7.043397e+08,8.226397e+08,1.061200e+09,...,2.095841e+10,2.587360e+10,2.631151e+10,2.330867e+10,1.813776e+10,2.209642e+10,2.916378e+10,2.757796e+10,2.530319e+10,NaN


In [99]:
gdp_by_entry = df.melt(id_vars=['Country Name','Country Code','Indicator Name','Indicator Code'], var_name='Year', value_name='GDP ($USD)')
gdp_by_entry['Year'] = pd.to_numeric(gdp_by_entry['Year'], errors='coerce')
gdp_by_entry.drop(columns=['Indicator Name', 'Indicator Code'], inplace=True)
gdp_by_entry

,Country Name,Country Code,Year,GDP ($USD)
0,Aruba,ABW,1960,NaN
1,Africa Eastern and Southern,AFE,1960,2.420569e+10
2,Afghanistan,AFG,1960,NaN
3,Africa Western and Central,AFW,1960,1.190481e+10
4,Angola,AGO,1960,NaN
...,...,...,...,...
17551,Kosovo,XKX,2025,NaN
17552,"Yemen, Rep.",YEM,2025,NaN
17553,South Africa,ZAF,2025,NaN
17554,Zambia,ZMB,2025,NaN


In [100]:
# Merge country metadata and create grouped DataFrames
meta_country = pd.read_csv('Metadata_Country_API_NY.GDP.MKTP.CD_DS2_en_csv_v2_133326.csv')

gdp_meta = gdp_by_entry.merge(meta_country[['Country Code','Region','IncomeGroup','TableName']], on='Country Code', how='left')
gdp_meta.rename(columns={'IncomeGroup':'Income Group'}, inplace=True)
gdp_meta.drop(columns=['TableName'], inplace=True)

In [101]:
# By-region time series (sum of GDP per year per region)

# Use the long-format dataframe for Tableau or Plotly
by_region = gdp_meta.groupby(['Region','Year'])['GDP ($USD)'].sum().reset_index()

# Output a CSV for Tableau
by_region.to_csv('gdp_by_region.csv', index=False)

# Use the wide-format dataframe for Matplotlib
by_region_pivot = by_region.pivot(index='Year', columns='Region', values='GDP ($USD)')

by_region


,Region,Year,GDP ($USD)
0,East Asia & Pacific,1960,1.505167e+11
1,East Asia & Pacific,1961,1.515142e+11
2,East Asia & Pacific,1962,1.549348e+11
3,East Asia & Pacific,1963,1.728853e+11
4,East Asia & Pacific,1964,1.984405e+11
...,...,...,...
457,Sub-Saharan Africa,2021,2.133014e+12
458,Sub-Saharan Africa,2022,2.284276e+12
459,Sub-Saharan Africa,2023,2.106924e+12
460,Sub-Saharan Africa,2024,1.968899e+12


In [102]:
# By-income-group time series
by_income = gdp_meta.groupby(['Income Group','Year'])['GDP ($USD)'].sum().reset_index()
by_income.to_csv('gdp_by_income_group.csv', index=False)

by_income_pivot = by_income.pivot(index='Year', columns='Income Group', values='GDP ($USD)')
by_income_pivot.head()


Income Group,High income,Low income,Lower middle income,Upper middle income
Year,,,,
1960,1.033979e+12,8.824431e+09,7.606060e+10,1.475997e+11
1961,1.102732e+12,8.895736e+09,8.210554e+10,1.455128e+11
1962,1.194266e+12,1.006390e+10,8.337784e+10,1.468288e+11
1963,1.284433e+12,1.284999e+10,9.224052e+10,1.613887e+11
1964,1.403686e+12,9.974091e+09,1.036444e+11,1.849908e+11


In [103]:
# If the region is missing, it's not a country, so we can drop those entries for the country-level dataset
by_country = gdp_meta.dropna(subset=['Region']).reset_index(drop=True)
by_country.to_csv('gdp_by_country.csv', index=False)
by_country

,Country Name,Country Code,Year,GDP ($USD),Region,Income Group
0,Aruba,ABW,1960,NaN,Latin America & Caribbean,High income
1,Afghanistan,AFG,1960,NaN,Middle East & North Africa,Low income
2,Angola,AGO,1960,NaN,Sub-Saharan Africa,Lower middle income
3,Albania,ALB,1960,NaN,Europe & Central Asia,Upper middle income
4,Andorra,AND,1960,NaN,Europe & Central Asia,High income
...,...,...,...,...,...,...
14317,Kosovo,XKX,2025,NaN,Europe & Central Asia,Upper middle income
14318,"Yemen, Rep.",YEM,2025,NaN,Middle East & North Africa,Low income
14319,South Africa,ZAF,2025,NaN,Sub-Saharan Africa,Upper middle income
14320,Zambia,ZMB,2025,NaN,Sub-Saharan Africa,Lower middle income
